# 01 — Exploration des gestes MPU6050

Analyse exploratoire des CSV capturés (`vertical`, `horizontal`, `circle`).

**Répertoire de travail :** racine du dépôt (`ia-iot/`).

```bash
# depuis la racine
jupyter notebook notebooks/01_eda.ipynb
# ou
jupyter lab notebooks/01_eda.ipynb
```


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Racine du dépôt (notebooks/ -> parent)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "python"))

from gesture_dataset import (
    EXPECTED_SAMPLES_PER_TRIAL,
    LABELS,
    SAMPLE_RATE_HZ,
    SENSOR_COLS,
    load_gesture_csvs,
    trial_summary,
)

RAW_DIR = ROOT / "data" / "raw"
print("ROOT =", ROOT)
print("RAW_DIR =", RAW_DIR)


## Chargement des CSV

Fichiers attendus : `data/raw/vertical.csv`, `horizontal.csv`, `circle.csv`.


In [ ]:
df = load_gesture_csvs(RAW_DIR)
print(f"Lignes totales : {len(df):,}")
print(f"Colonnes : {list(df.columns)}")
df.head()


## Comptage d'essais et équilibre des classes


In [ ]:
summary = trial_summary(df)
counts = summary.groupby("label")["trial_id"].count().reindex(LABELS)
print("Essais par classe :")
display(counts.to_frame("n_trials"))
print(f"Total essais : {int(counts.sum())}")

ax = counts.plot(kind="bar", color=["#2a6f97", "#ee6c4d", "#3d5a80"], rot=0, figsize=(6, 3.5))
ax.set_ylabel("Nombre d'essais")
ax.set_title("Équilibre des classes")
ax.set_xlabel("")
plt.tight_layout()
plt.show()


## Fréquence d'échantillonnage et longueur des essais

Attendu : **75** échantillons / essai à **50 Hz** (durée 1,5 s).


In [ ]:
print("Échantillons / essai — stats :")
display(summary["n_samples"].describe())

bad = summary[summary["n_samples"] != EXPECTED_SAMPLES_PER_TRIAL]
print(f"Essais ≠ {EXPECTED_SAMPLES_PER_TRIAL} échantillons : {len(bad)}")
if len(bad):
    display(bad.head(20))

print("\nFréquence estimée (Hz) :")
display(summary["sample_rate_hz"].describe())
print(f"Cible nominale : {SAMPLE_RATE_HZ} Hz")


## Statistiques descriptives des capteurs


In [ ]:
print(df[list(SENSOR_COLS)].describe().T)
print("\nValeurs manquantes :")
print(df[list(SENSOR_COLS)].isna().sum())


## Tracés d'exemples par classe

Un essai représentatif (le premier) pour chaque geste — accélération et gyroscope.


In [ ]:
fig, axes = plt.subplots(len(LABELS), 2, figsize=(12, 8), sharex=True)
if len(LABELS) == 1:
    axes = np.array([axes])

for i, label in enumerate(LABELS):
    trials = summary.loc[summary["label"] == label, "trial_id"]
    if trials.empty:
        axes[i, 0].set_title(f"{label} — aucun essai")
        continue
    tid = trials.iloc[0]
    g = df[df["trial_id"] == tid].sort_values("sample_index")
    t = g["timestamp_ms"]

    for col in ("ax", "ay", "az"):
        axes[i, 0].plot(t, g[col], label=col, lw=1)
    axes[i, 0].set_ylabel(label)
    axes[i, 0].set_title(f"{label} — accélération ({tid})")
    axes[i, 0].legend(loc="upper right", fontsize=8)

    for col in ("gx", "gy", "gz"):
        axes[i, 1].plot(t, g[col], label=col, lw=1)
    axes[i, 1].set_title(f"{label} — gyroscope ({tid})")
    axes[i, 1].legend(loc="upper right", fontsize=8)

axes[-1, 0].set_xlabel("timestamp_ms")
axes[-1, 1].set_xlabel("timestamp_ms")
plt.tight_layout()
plt.show()


## Superposition de plusieurs essais (axe dominant)

Pour visualiser la variabilité intra-classe, on trace `|a|` (norme accélération) pour jusqu'à 8 essais.


In [ ]:
fig, axes = plt.subplots(1, len(LABELS), figsize=(12, 3.5), sharey=True)
if len(LABELS) == 1:
    axes = [axes]

for ax, label in zip(axes, LABELS):
    trials = summary.loc[summary["label"] == label, "trial_id"].head(8)
    for tid in trials:
        g = df[df["trial_id"] == tid].sort_values("sample_index")
        amag = np.sqrt(g["ax"] ** 2 + g["ay"] ** 2 + g["az"] ** 2)
        ax.plot(g["timestamp_ms"], amag, alpha=0.7, lw=1)
    ax.set_title(label)
    ax.set_xlabel("timestamp_ms")

axes[0].set_ylabel("|a| (brut)")
plt.suptitle("Norme accélération — plusieurs essais", y=1.02)
plt.tight_layout()
plt.show()


## Sessions de capture


In [ ]:
if "session" in df.columns:
    sess = (
        df.groupby(["label", "session"])["trial_id"]
        .nunique()
        .rename("n_trials")
        .reset_index()
    )
    display(sess)
else:
    print("Colonne session absente.")


## Conclusion EDA

- Vérifier l'équilibre des classes et les essais incomplets avant l'entraînement.
- Passer au notebook `02_train_eval.ipynb` pour features + classification.
